In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import os
import glob

# ================================
# Clase HiddenNaiveBayes (sin visualización del grafo)
# ================================
class HiddenNaiveBayes:
    def __init__(self, epsilon=1e-6):
        self.class_probs = {}
        self.cond_probs = defaultdict(dict)
        self.weights = defaultdict(dict)
        self.mutual_info = defaultdict(dict)
        self.hidden_parents = defaultdict(list)
        self.epsilon = epsilon

    def _calculate_class_probabilities(self, y):
        classes, counts = np.unique(y, return_counts=True)
        total_samples = len(y)
        k = len(classes)
        for c, count in zip(classes, counts):
            self.class_probs[c] = (count + self.epsilon / k) / (total_samples + self.epsilon)

    def _calculate_conditional_probabilities(self, X, y):
        n_samples, n_features = X.shape
        classes = np.unique(y)
        for c in classes:
            class_filter = (y == c)
            for i in range(n_features):
                feature_values, counts = np.unique(X[class_filter, i], return_counts=True)
                total = np.sum(counts)
                n_i = len(feature_values)
                # Cálculo de P(a_i | c) con M-estimación
                self.cond_probs[(i, c)] = {val: (count + self.epsilon / n_i) / (total + self.epsilon)
                                           for val, count in zip(feature_values, counts)}
                # Cálculo de P(a_i | a_j, c) para cada par (i, j)
                for j in range(n_features):
                    if i != j:
                        pair_counts = defaultdict(int)
                        for xi, xj in zip(X[class_filter, i], X[class_filter, j]):
                            pair_counts[(xi, xj)] += 1
                        total_pairs = sum(pair_counts.values())
                        n_ij = len(pair_counts)
                        self.cond_probs[(i, j, c)] = {pair: (count + self.epsilon / n_ij) / (total_pairs + self.epsilon)
                                                      for pair, count in pair_counts.items()}

    def _calculate_mutual_information(self, X, y):
        classes = np.unique(y)
        n_features = X.shape[1]
        for c in classes:
            for i in range(n_features):
                for j in range(n_features):
                    if i != j:
                        mi_sum = 0
                        for (a_i, a_j), p_aij_c in self.cond_probs[(i, j, c)].items():
                            p_ai_c = self.cond_probs[(i, c)].get(a_i, self.epsilon)
                            p_aj_c = self.cond_probs[(j, c)].get(a_j, self.epsilon)
                            mi_sum += p_aij_c * np.log(p_aij_c / (p_ai_c * p_aj_c))
                        self.mutual_info[(i, j, c)] = mi_sum

    def _calculate_weights(self, X, y):
        classes = np.unique(y)
        n_features = X.shape[1]
        for c in classes:
            for i in range(n_features):
                mi_sum = sum(self.mutual_info[(i, k, c)] for k in range(n_features) if k != i)
                for j in range(n_features):
                    if i != j:
                        self.weights[(i, j, c)] = (self.mutual_info[(i, j, c)] / mi_sum) if mi_sum != 0 else 0
                        # Se establece un umbral para determinar padres ocultos
                        if self.weights[(i, j, c)] > 0.15:
                            self.hidden_parents[i].append(j)

    def fit(self, X, y):
        self._calculate_class_probabilities(y)
        self._calculate_conditional_probabilities(X, y)
        self._calculate_mutual_information(X, y)
        self._calculate_weights(X, y)

    def _calculate_hidden_parent(self, a_i, i, X_sample, c):
        weighted_sum = 0
        for j in range(len(X_sample)):
            if i != j:
                a_j = X_sample[j]
                weighted_sum += self.weights.get((i, j, c), 0) * \
                                self.cond_probs.get((i, j, c), {}).get((a_i, a_j), self.epsilon)
        return max(weighted_sum, self.epsilon)

    def predict(self, X):
        predictions = []
        for X_sample in X:
            class_scores = {}
            for c in self.class_probs:
                score = np.log(self.class_probs[c])
                for i in range(X_sample.shape[0]):
                    a_i = X_sample[i]
                    p_a_i_given_h = self._calculate_hidden_parent(a_i, i, X_sample, c)
                    score += np.log(p_a_i_given_h)
                class_scores[c] = score
            best_class = max(class_scores, key=class_scores.get)
            predictions.append(best_class)
        return predictions

    def predict_probabilities(self, X):
        probabilities = []
        all_classes = sorted(self.class_probs.keys())
        for X_sample in X:
            class_probs = {cls: 0 for cls in all_classes}
            for c in self.class_probs:
                score = np.log(self.class_probs[c])
                for i in range(X_sample.shape[0]):
                    a_i = X_sample[i]
                    p_a_i_given_h = self._calculate_hidden_parent(a_i, i, X_sample, c)
                    score += np.log(p_a_i_given_h)
                class_probs[c] = np.exp(score)
            total_prob = sum(class_probs.values())
            class_probs = {k: v / total_prob for k, v in class_probs.items()}
            probabilities.append([class_probs[cls] for cls in all_classes])
        return np.array(probabilities)

# =====================================
# Función para validación cruzada simple (sólo accuracy)
# =====================================
def hnb_simple_cross_validate(data, target, k=10):
    # Mezclar y dividir el dataset en k particiones
    folds = np.array_split(data.sample(frac=1, random_state=42), k)
    accuracies = []

    for i in range(k):
        # Crear conjunto de entrenamiento y de prueba
        train = pd.concat([folds[j] for j in range(k) if j != i], ignore_index=True)
        test = folds[i]

        # Entrenar el modelo Hidden Naive Bayes
        hnb_model = HiddenNaiveBayes()
        X_train = train.drop(target, axis=1).values
        y_train = train[target].values
        hnb_model.fit(X_train, y_train)

        # Predecir sobre el conjunto de prueba
        X_test = test.drop(target, axis=1).values
        y_test = test[target].values
        predictions = hnb_model.predict(X_test)
        accuracy = np.mean(predictions == y_test)
        accuracies.append(accuracy)

    return np.mean(accuracies), np.std(accuracies)

# =====================================
# Procesar múltiples archivos CSV en una carpeta y guardar resultados en un DataFrame
# =====================================

# Ruta de la carpeta que contiene los archivos CSV
carpeta_bd = r"C:\Users\Carlo\Desktop\IA\MDLP\base de datos discretizadas con mdlp R"

# Buscar todos los archivos CSV en la carpeta
archivos_csv = glob.glob(os.path.join(carpeta_bd, "*.csv"))

resultados = []

for archivo in archivos_csv:
    try:
        data = pd.read_csv(archivo)
        # Se asume que la última columna es la variable objetivo
        target = data.columns[-1]
        mean_acc, std_acc = hnb_simple_cross_validate(data, target=target, k=10)
        resultados.append({
            "Dataset": os.path.basename(archivo),
            "Mean Accuracy": mean_acc,
            "Std Accuracy": std_acc
        })
        print(f"Procesado {os.path.basename(archivo)}: Precisión media = {mean_acc:.6f}, Desviación = {std_acc:.6f}")
    except Exception as e:
        print(f"Error al procesar {os.path.basename(archivo)}: {e}")

# Crear un DataFrame con los resultados y mostrarlo
df_resultados = pd.DataFrame(resultados)
print("\nResultados finales:")
print(df_resultados)


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado dry_bean_mdlpR.csv: Precisión media = 0.905297, Desviación = 0.004983
Procesado glass_mdlpR.csv: Precisión media = 0.471429, Desviación = 0.084813
Procesado iris_mdlpR.csv: Precisión media = 0.873333, Desviación = 0.172434


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado letter_recognition_mdlpR.csv: Precisión media = 0.526800, Desviación = 0.013108


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


...

Procesado winequality-red_mdlpR.csv: Precisión media = 0.553443, Desviación = 0.040267


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado winequality-white_mdlpR.csv: Precisión media = 0.459783, Desviación = 0.012713


c:\Users\Carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Procesado yeast_mdlpR.csv: Precisión media = 0.345583, Desviación = 0.078196

Resultados finales:
                        Dataset  Mean Accuracy  Std Accuracy
0            dry_bean_mdlpR.csv       0.905297      0.004983
1               glass_mdlpR.csv       0.471429      0.084813
2                iris_mdlpR.csv       0.873333      0.172434
3  letter_recognition_mdlpR.csv       0.526800      0.013108
4                rice_mdlpR.csv       0.928871      0.010593
5               seeds_mdlpR.csv       0.857143      0.060234
6     winequality-red_mdlpR.csv       0.553443      0.040267
7   winequality-white_mdlpR.csv       0.459783      0.012713
8               yeast_mdlpR.csv       0.345583      0.078196


Resultados de mi mdlp:

Resultados finales:
                       Dataset  Mean Accuracy  Std Accuracy
0            dry_bean_mdlp.csv       0.906032      0.005216
1               gamma_mdlp.csv       0.799159      0.004596
2               glass_mdlp.csv       0.467100      0.104538
3                iris_mdlp.csv       0.586667      0.097980
4  letter_recognition_mdlp.csv       0.531250      0.013340
5          rice_cameo_mdlp.csv       0.928871      0.010128
6               seeds_mdlp.csv       0.633333      0.085317
7     winequality-red_mdlp.csv       0.567197      0.046493
8   winequality-white_mdlp.csv       0.459783      0.013135
9               yeast_mdlp.csv       0.584215      0.026374

Resultados con CAIM 

Resultados finales:
                Dataset  Mean Accuracy  Std Accuracy
0     dry-bean_caim.csv       0.895453      0.007192
1      dryBean_caim.csv       0.428992      0.010698
2        gamma_caim.csv       0.798107      0.008813
3        glass_caim.csv       0.714935      0.116791
4         iris_caim.csv       0.853333      0.151438
5       letter_caim.csv       0.817800      0.009811
6         rice_caim.csv       0.914961      0.009974
7        seeds_caim.csv       0.804762      0.133758
8     wine-red_caim.csv       0.549104      0.021137
9   wine-white_caim.csv       0.509803      0.015987
10       yeast_caim.csv       0.371227      0.053980

Resultados finales con mdlp R:

Resultados finales:
                        Dataset  Mean Accuracy  Std Accuracy
0            dry_bean_mdlpR.csv       0.905297      0.004983
1               glass_mdlpR.csv       0.471429      0.084813
2                iris_mdlpR.csv       0.873333      0.172434
3  letter_recognition_mdlpR.csv       0.526800      0.013108
4                rice_mdlpR.csv       0.928871      0.010593
5               seeds_mdlpR.csv       0.857143      0.060234
6     winequality-red_mdlpR.csv       0.553443      0.040267
7   winequality-white_mdlpR.csv       0.459783      0.012713
8               yeast_mdlpR.csv       0.345583      0.078196